## Install Packages

In [3]:
!pip install -q datasets huggingface_hub

## Login to hugging face

In [2]:
from huggingface_hub import login

login()

# One time HF upload

## Download CIFAR-10

In [3]:
import os
from torchvision.datasets import CIFAR10

DATA_DIR = "/content/cifar-10"

os.makedirs(DATA_DIR, exist_ok=True)

train_dataset = CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True
)

test_dataset = CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True
)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

100%|██████████| 170M/170M [25:23<00:00, 112kB/s]


Train samples: 50000
Test samples: 10000


## Verify download

In [4]:
import os

for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")

cifar-10/
  cifar-10-python.tar.gz
  cifar-10-batches-py/
    data_batch_3
    data_batch_4
    data_batch_5
    test_batch
    readme.html
    data_batch_1
    data_batch_2
    batches.meta


## Verify CIFAR-10

In [5]:
assert len(train_dataset) == 50_000
assert len(test_dataset) == 10_000

print("✓ CIFAR-10 verified")
print(f"✓ Training samples: {len(train_dataset):,}")
print(f"✓ Test samples:     {len(test_dataset):,}")
print(f"✓ Total samples:    {len(train_dataset) + len(test_dataset):,}")
print(f"✓ Classes:          {train_dataset.classes}")

✓ CIFAR-10 verified
✓ Training samples: 50,000
✓ Test samples:     10,000
✓ Total samples:    60,000
✓ Classes:          ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Upload on Hugging Face

In [6]:
from huggingface_hub import HfApi

REPO_ID = "Sakhiur/blue-bird-77"

api = HfApi()

api.upload_folder(
    folder_path=DATA_DIR,
    path_in_repo="cifar-10",
    repo_id=REPO_ID,
    repo_type="dataset",
)

print("✓ CIFAR-10 uploaded successfully")
print(f"https://huggingface.co/datasets/{REPO_ID}/tree/main/cifar-10")

✓ CIFAR-10 uploaded successfully
https://huggingface.co/datasets/Sakhiur/blue-bird-77/tree/main/cifar-10


## Verify Remote

In [7]:
from huggingface_hub import list_repo_tree

files = list(
    list_repo_tree(
        REPO_ID,
        repo_type="dataset",
        path_in_repo="cifar-10"
    )
)

for file in files:
    print(file.path)

cifar-10/cifar-10-batches-py
cifar-10/cifar-10-python.tar.gz


# Main Stuff

## From HF

In [1]:
from huggingface_hub import snapshot_download

REPO_ID = "Sakhiur/blue-bird-77"

local_dir = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns="cifar-10/*"
)

print("Downloaded to:")
print(local_dir)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded to:
/root/.cache/huggingface/hub/datasets--Sakhiur--blue-bird-77/snapshots/5bd2da09cc5b107ffb093d9c1920ba7a729bf587


## Load CIFAR-10

In [8]:
import os
import pickle
import numpy as np
from PIL import Image

CIFAR_DIR = os.path.join(local_dir, "cifar-10", "cifar-10-batches-py")

def load_batch(path):
    with open(path, "rb") as f:
        batch = pickle.load(f, encoding="bytes")

    images = batch[b"data"]
    labels = batch[b"labels"]

    # CIFAR-10 stores images as:
    # 3072 = 3 channels × 32 × 32
    images = images.reshape(-1, 3, 32, 32)
    images = np.transpose(images, (0, 2, 3, 1))

    return images, np.array(labels)


# Load training batches
train_images = []
train_labels = []

for i in range(1, 6):
    images, labels = load_batch(
        os.path.join(CIFAR_DIR, f"data_batch_{i}")
    )

    train_images.append(images)
    train_labels.append(labels)


train_images = np.concatenate(train_images)
train_labels = np.concatenate(train_labels)


# Load test batch
test_images, test_labels = load_batch(
    os.path.join(CIFAR_DIR, "test_batch")
)

print("Train:", train_images.shape, train_labels.shape)
print("Test :", test_images.shape, test_labels.shape)

Train: (50000, 32, 32, 3) (50000,)
Test : (10000, 32, 32, 3) (10000,)


## Define CIFAR-10 Class

In [9]:
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

print(CLASS_NAMES)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Quick Verification

In [10]:
assert train_images.shape == (50_000, 32, 32, 3)
assert train_labels.shape == (50_000,)

assert test_images.shape == (10_000, 32, 32, 3)
assert test_labels.shape == (10_000,)

assert train_labels.min() == 0
assert train_labels.max() == 9

assert test_labels.min() == 0
assert test_labels.max() == 9

print(" CIFAR-10 loaded successfully from Hugging Face")
print(f"Training samples: {len(train_images):,}")
print(f"Test samples:     {len(test_images):,}")
print(f"Image size:       {train_images.shape[1:]}")
print(f"Classes:          {len(CLASS_NAMES)}")

 CIFAR-10 loaded successfully from Hugging Face
Training samples: 50,000
Test samples:     10,000
Image size:       (32, 32, 3)
Classes:          10


## Interactive CIFAR-10 inspector

In [13]:
import random
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

split_dropdown = widgets.Dropdown(
    options=["Train", "Test"],
    value="Train",
    description="Split:"
)

class_dropdown = widgets.Dropdown(
    options=["All"] + CLASS_NAMES,
    value="All",
    description="Class:"
)

num_slider = widgets.IntSlider(
    value=12,
    min=1,
    max=36,
    description="Images:"
)

seed_slider = widgets.IntSlider(
    value=42,
    min=0,
    max=9999,
    description="Seed:"
)

button = widgets.Button(
    description="Show Images",
    button_style="primary"
)

output = widgets.Output()


def inspect_cifar(_=None):

    with output:
        clear_output(wait=True)

        if split_dropdown.value == "Train":
            images = train_images
            labels = train_labels
        else:
            images = test_images
            labels = test_labels

        selected_class = class_dropdown.value

        if selected_class == "All":
            indices = np.arange(len(images))
        else:
            class_id = CLASS_NAMES.index(selected_class)
            indices = np.where(labels == class_id)[0]

        random.seed(seed_slider.value)

        indices = random.sample(
            list(indices),
            min(num_slider.value, len(indices))
        )

        cols = 6
        rows = (len(indices) + cols - 1) // cols

        fig, axes = plt.subplots(
            rows,
            cols,
            figsize=(15, 2.8 * rows)
        )

        axes = np.array(axes).reshape(-1)

        for ax, idx in zip(axes, indices):
            ax.imshow(images[idx])
            ax.set_title(
                f"{CLASS_NAMES[labels[idx]]}\nindex={idx}"
            )
            ax.axis("off")

        for ax in axes[len(indices):]:
            ax.axis("off")

        plt.tight_layout()
        plt.show()


button.on_click(inspect_cifar)

display(
    widgets.VBox([
        split_dropdown,
        class_dropdown,
        num_slider,
        seed_slider,
        button,
        output
    ])
)

inspect_cifar()

## Individual Inspector

In [16]:
split_widget = widgets.Dropdown(
    options=["Train", "Test"],
    value="Train",
    description="Split:"
)

index_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=len(train_images) - 1,
    step=1,
    description="Index:",
    continuous_update=False
)

previous_button = widgets.Button(
    description="← Previous"
)

next_button = widgets.Button(
    description="Next →"
)

random_button = widgets.Button(
    description="🎲 Random"
)

output = widgets.Output()



# Get selected dataset


def get_dataset():

    if split_widget.value == "Train":
        return train_images, train_labels

    return test_images, test_labels



# Display image


def show_sample():

    with output:
        clear_output(wait=True)

        images, labels = get_dataset()

        idx = index_widget.value

        image = images[idx]
        label = labels[idx]

        plt.figure(figsize=(5, 5))
        plt.imshow(image)

        plt.title(
            f"{split_widget.value}\n"
            f"Index {idx} — {CLASS_NAMES[label]}"
        )

        plt.axis("off")
        plt.show()

        print(f"Index:       {idx}")
        print(f"Label ID:    {label}")
        print(f"Class:       {CLASS_NAMES[label]}")
        print(f"Image shape: {image.shape}")
        print(f"Data type:   {image.dtype}")



# Previous


def previous(_):

    if index_widget.value > 0:
        index_widget.value -= 1

    show_sample()



# Next


def next_sample(_):

    if index_widget.value < index_widget.max:
        index_widget.value += 1

    show_sample()



# Random

def random_sample(_):

    import random

    index_widget.value = random.randint(
        0,
        index_widget.max
    )

    show_sample()



# Split changed


def split_changed(change):

    if change["new"] == "Train":
        index_widget.max = len(train_images) - 1
    else:
        index_widget.max = len(test_images) - 1

    index_widget.value = 0

    show_sample()



# Connect buttons


previous_button.on_click(previous)
next_button.on_click(next_sample)
random_button.on_click(random_sample)

split_widget.observe(
    split_changed,
    names="value"
)



# Display UI


display(
    widgets.VBox([
        widgets.HTML("<h3> CIFAR-10 Individual Inspector</h3>"),

        split_widget,

        index_widget,

        widgets.HBox([
            previous_button,
            next_button,
            random_button
        ]),

        output
    ])
)

show_sample()